In [ ]:
#
# ⚡ UNIVERSAL FIRST CELL - Run this FIRST in every notebook!
# Compatible with: Google Colab, GitHub Codespaces, Local
#

import os
import subprocess
import sys

# Detect environment
IS_COLAB = "google.colab" in sys.modules
IS_CODESPACES = os.path.exists("/.devcontainer") or os.path.exists("/workspaces")
IS_LOCAL = not (IS_COLAB or IS_CODESPACES)

print(f"🚀 Environment: {'Colab' if IS_COLAB else 'Codespaces' if IS_CODESPACES else 'Local'}")

# Clone repo in Colab if needed
if IS_COLAB and not os.path.exists("computer-data-analysis-report"):
    print("Cloning repository...")
    !git clone --depth 1 https://github.com/Aidas-dev/computer-data-analysis-report.git

# Change to repo directory
repo_dir = "computer-data-analysis-report"
if os.path.exists(repo_dir):
    %cd {repo_dir}
    !git pull -q
else:
    !cd {repo_dir} 2>/dev/null || echo "Repo not found"

# Install uv (faster than pip)
!pip install uv -q

# Install dependencies with uv (bypasses Colab's pip resolver issues)
!uv pip install --system -r requirements.txt -q

# Setup DVC in Colab
if IS_COLAB:
    from google.colab import userdata
    try:
        access_key = userdata.get("OCI_ACCESS_KEY")
        secret_key = userdata.get("OCI_SECRET_KEY")
        !dvc remote modify --local oracle_remote access_key_id {access_key}
        !dvc remote modify --local oracle_remote secret_access_key {secret_key}
        !dvc pull -q
    except:
        pass  # Secrets not configured, skip DVC pull

print("✅ Environment ready!")


# Google Colab Setup for Data Analysis Project

**🚨 IMPORTANT: You must run this notebook at the beginning of EVERY new Google Colab session!**
This ensures the public repository is cloned, dependencies are installed, and the private data is securely pulled from the Oracle Cloud bucket.

**Before you run this notebook:**
1. Click the **🔑 Secrets** icon on the left sidebar.
2. Add two new secrets:
   - Name: `OCI_ACCESS_KEY` | Value: `YOUR_ACCESS_KEY`
   - Name: `OCI_SECRET_KEY` | Value: `YOUR_SECRET_KEY`
   
*(Make sure the toggle for "Notebook access" is checked for both secrets).*

In [ ]:
import os

# 1. Clone the public repository
repo_url = "https://github.com/Aidas-dev/computer-data-analysis-report.git"
repo_name = "computer-data-analysis-report"

# Use Colab's magic command to change directory for both Python and the shell
if os.path.exists("/content"):
    %cd /content

if not os.path.exists(repo_name):
    print(f"Cloning {repo_url}...")
    !git clone {repo_url}

if os.path.exists(f"/content/{repo_name}"):
    %cd /content/{repo_name}
else:
    %cd {repo_name}

print("Pulling latest changes...")
!git pull

# 2. Install dependencies (using uv to fix Colab's pip conflicts and speed up install)
print("Installing dependencies with uv...")
!pip install uv -q
!uv pip install --system -r requirements.txt -q

In [ ]:
from google.colab import userdata
import os

try:
    print("Authenticating DVC securely using Colab Secrets...")
    access_key = userdata.get("OCI_ACCESS_KEY")
    secret_key = userdata.get("OCI_SECRET_KEY")
    
    # Configure DVC locally within the Colab environment
    !dvc remote modify --local oracle_remote access_key_id {access_key}
    !dvc remote modify --local oracle_remote secret_access_key {secret_key}
    
    # 3. Pull data
    print("Pulling private data from Oracle Cloud Object Storage...")
    !dvc pull
    print("✅ Setup complete! Data pulled successfully and ready for analysis.")
except userdata.SecretNotFoundError as e:
    print(f"❌ Secret not found: {e}. Please add it to the Colab Secrets manager (🔑 icon on the left).")
except Exception as e:
    print(f"❌ An error occurred: {e}")